# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jasleen13/ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task

*What would you predict? Where does that label come from observed outcome or a defined rule?*

**Ranking / scoring.** Lane 4's question is "which ones first?"  given limited editor review
capacity, which pages should be reviewed before the others? That maps directly to a priority
score used to rank, not a yes/no classification. There's no natural binary event here ("this
page is broken" isn't observed in the data)  what's observed is a continuous CTR gap, which a
score preserves and a classifier would have to flatten into an arbitrary cutoff.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/jasleen13/ML-Internship.git"
DATA_REL = Path("data") / "raw" / "content_refresh_anonymized.csv"

def find_repo_root(start: Path):
    p = start
    while not (p / DATA_REL).exists() and p != p.parent:
        p = p.parent
    return p if (p / DATA_REL).exists() else None

repo_root = find_repo_root(Path.cwd())

if repo_root is None:
    clone_dir = Path("/content/ML-Internship") if Path("/content").exists() else Path.cwd() / "ML-Internship"
    if not (clone_dir / DATA_REL).exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

DATA_PATH = repo_root / DATA_REL
print("Using data at:", DATA_PATH)
assert DATA_PATH.exists(), "Data file still not found, check REPO_URL and your connection."


df = pd.read_csv(DATA_PATH)
print(df.shape)


Using data at: /content/ML-Internship/data/raw/content_refresh_anonymized.csv
(30000, 44)


## 2. Target or proxy

Proxy: ctr_gap, a page's observed ctr minus the median ctr of other pages in the same position_tier (both computed from the same trailing-90-day window). A negative gap means a page under-captures clicks relative to peers at a similar rank.

This is a derived measurement from observed signals, not a rule-based product flag. Per the flyrank-data skill, ctr is a real measurement (clicks_90d / impressions_90d x 100), and the tier median is computed only from the current feature window, so nothing here reaches into the future or copies a FlyRank product decision like health_score.

Being honest about what kind of target this is: it's a current-window proxy, not an observed future outcome. It tells me a page currently sits below its tier's peers. It does not tell me whether that page would improve if reviewed, and it can't, without a before/after experiment. If a later week turns this into a classifier ("will CTR rise after a rewrite"), that would need a strictly future target window and a leakage audit, out of scope for this lane as scoring only.

In [12]:
# Lane slice: pages with enough volume to trust their CTR, and a real position reading
lane = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

tier_expected_ctr = lane.groupby("position_tier")["ctr"].median()
print("Tier expected CTR (median, volume-floored):")
print(tier_expected_ctr)

lane["tier_expected_ctr"] = lane["position_tier"].map(tier_expected_ctr)
lane["ctr_gap"] = lane["ctr"] - lane["tier_expected_ctr"]
print()
print("ctr_gap distribution:")
print(lane["ctr_gap"].describe())


Tier expected CTR (median, volume-floored):
position_tier
deep        0.00
page_1      0.23
page_3_5    0.06
striking    0.15
top_3       0.19
Name: ctr, dtype: float64

ctr_gap distribution:
count    22006.000000
mean         0.105696
std          0.391143
min         -0.230000
25%         -0.070000
50%          0.000000
75%          0.180000
max         11.530000
Name: ctr_gap, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K on a manual read of the top-ranked queue**, e.g. precision@50: of the 50
lowest-`ctr_gap` pages the score surfaces first, how many hold up as genuine review candidates
(enough volume, real position, gap large enough to plausibly be signal, not noise) when I
actually look at them by hand? This matches how the output gets used: an editor works down a
ranked list with limited time, so the top of the list is what has to be right, not the whole
ranking.

I'll also sanity-check the score against the naive alternative, a single global CTR threshold
with no position adjustment, by comparing how evenly each pulls candidates across tiers (a
good score shouldn't systematically over-flag one tier just because its raw CTR numbers run
lower).


In [13]:
# Preview: the queue sorted by ctr_gap (most negative first = biggest under-performers)
review_queue = lane.sort_values("ctr_gap").head(20)
print(review_queue[["content_id", "position_tier", "impressions_90d", "avg_position", "ctr",
                     "tier_expected_ctr", "ctr_gap"]].to_string(index=False))

          content_id position_tier  impressions_90d  avg_position  ctr  tier_expected_ctr  ctr_gap
content_2a228ce7aa1b        page_1              155           7.5  0.0               0.23    -0.23
content_5ef76f383560        page_1              973           3.2  0.0               0.23    -0.23
content_12018e6ac413        page_1              144           8.3  0.0               0.23    -0.23
content_3bc49e1805db        page_1              683           6.8  0.0               0.23    -0.23
content_b0bbe3291c4c        page_1              106           7.3  0.0               0.23    -0.23
content_9f8ff0fcf40b        page_1              262           3.3  0.0               0.23    -0.23
content_4a46bc087129        page_1              678           5.1  0.0               0.23    -0.23
content_212375a53741        page_1             1258           3.2  0.0               0.23    -0.23
content_10537c63f996        page_1              370           3.5  0.0               0.23    -0.23
content_10

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page** (`content_id`), scoped to pages with enough impression volume to trust
their CTR and a real (non-zero) position reading. The dataframe below is that slice, with the
scoring columns attached.

In [14]:
print(f"Unit of analysis: one row = one page. Lane slice size: {len(lane):,} of {len(df):,} total pages")
lane[["content_id", "client_id", "position_tier", "impressions_90d", "avg_position",
      "ctr", "tier_expected_ctr", "ctr_gap"]].head(10)

Unit of analysis: one row = one page. Lane slice size: 22,006 of 30,000 total pages


,content_id,client_id,position_tier,impressions_90d,avg_position,ctr,tier_expected_ctr,ctr_gap
0,content_304f48230142,client_f369cb89fc,striking,3803,10.6,0.76,0.15,0.61
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,15320,20.3,0.05,0.06,-0.01
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,12581,36.5,0.09,0.06,0.03
3,content_331d6c4de07b,client_19581e27de,page_1,11751,6.2,0.49,0.23,0.26
4,content_d99b7a2d90ca,client_3fdba35f04,page_3_5,19140,44.0,0.13,0.06,0.07
5,content_d4084a4bc775,client_f369cb89fc,page_1,3970,8.5,0.03,0.23,-0.20
7,content_a63219c6e95a,client_19581e27de,page_3_5,1724,21.2,0.06,0.06,0.00
8,content_5e6c160719bc,client_6208ef0f77,page_3_5,32574,46.0,0.09,0.06,0.03
9,content_c27558df2b0c,client_19581e27de,page_1,1240,4.9,0.16,0.23,-0.07
10,content_d8ee6cc6d642,client_19581e27de,top_3,20919,2.2,1.55,0.19,1.36


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single global threshold (e.g. "flag anything with ctr < 0.2") ignores position, and the data shows exactly why that breaks: the same cutoff flags 44% of page_1 pages but 91% of deep pages, because deep-tier pages have low CTR by default, regardless of whether anything is actually wrong with them. A fixed rule can't express "compare each page only to its own peers" without hand-writing a separate threshold per tier, and even then, it collapses a continuous gap into a binary flag, throwing away the ranking information an editor actually needs to decide what to review first versus fifth versus fiftieth.

The tier-relative gap score isn't a full ML model yet (it's closer to a smart baseline), but it already captures what a fixed rule can't: position-normalized comparison and a continuous priority order. That's the seed a real model (Week 4+) would extend, e.g. learning tier boundaries, content-type effects, or seasonality adjustments the manual median can't pick up.

In [15]:
# Evidence: a naive fixed threshold flags very different shares of each tier,
# because it ignores that "low CTR" means something different at each position.
naive_flag_rate = lane.groupby("position_tier").apply(lambda g: (g["ctr"] < 0.2).mean())
print("Share of each tier flagged by a naive rule (ctr < 0.2), ignoring position:")
print(naive_flag_rate.sort_values(ascending=False))

Share of each tier flagged by a naive rule (ctr < 0.2), ignoring position:
position_tier
deep        0.908987
page_3_5    0.753219
striking    0.573098
top_3       0.510319
page_1      0.443646
dtype: float64


/tmp/ipykernel_620/3559843449.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  naive_flag_rate = lane.groupby("position_tier").apply(lambda g: (g["ctr"] < 0.2).mean())


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.